# Week 04 Day 02: Multidimensional Data Analysis of GitHub Repositories

This is a copy of `WK04_Day02_Data_Warehouse_v1.ipynb` adapted to the research proposal *Mining GitHub for Unmet Developer Needs*. It retains the lab's ten sections and executable steps while replacing the synthetic electronics sales data with the team's GitHub repository data from the previous lab. Every result describes the **2026-09-28 snapshot**. One row represents one repository, not a transaction or a time series. The proposal plans issue events and later activity data, but those are not present here.

**Objectives:** identify dimensions and measures, construct a star schema, compute cubes, perform roll-up, drill-down, slice, dice, and pivot, classify measures, compare OLAP designs, and evaluate repository cohorts. The source files are `../DM-Lab/github_repos_2019_2025.csv` and `../DM-Lab/github_repos_clean.csv`. Run cells in order from this notebook's folder. Requires pandas and ipykernel.

<a name="p1"></a>

---
# Section 01: Business Understanding
---

## Problem statement

The research project seeks recurring developer problems with many attempted solutions but poor maintenance. This warehouse organizes the available repository snapshot by creation time, language, owner type, license, and maintenance status. It can reveal where repositories, stars, and open issues are concentrated. It cannot establish an unmet need without the planned text clustering, issue evidence, and developer evaluation.

### Assess the situation

1. **Who are the primary stakeholders?** The research team, developers who review candidate needs, and software builders who may use the ranked report.
2. **What is stored in the operational source?** GitHub repository records: identifiers, timestamps, language, ownership, stars, forks, open issues, and other snapshot metadata. The proposal also plans issue and commit events, which are not in these CSVs.
3. **Why are strategic questions hard to answer from individual records?** Each record describes one repository. Comparisons across years, languages, and maintenance groups require consistent dimensions, joins, and aggregates.
4. **Three business questions?** Which creation cohorts have the most abandoned repositories? Which languages have the most open issues? Within a language and year, does maintenance differ by owner type?
5. **How could a warehouse improve decisions?** It makes those cohort summaries repeatable and keeps the repository grain and snapshot explicit.

### Business and analytical goals

The proposal's business goal is to rank candidate unmet developer needs with traceable evidence. This lab supports the data-understanding stage by summarizing the repository corpus.

1. **Which goal requires grouping by product?** The repository equivalent is grouping by `language` to compare ecosystems.
2. **Which goal requires grouping by region?** Group by `owner_type` or `license_group` to compare repository groups. There is no geographic region field.
3. **Which goal analyzes time?** Compare repository creation by year, quarter, and month.
4. **How do we move from yearly to quarterly counts?** Drill down.
5. **Why are aggregates useful?** They expose cohort size and status patterns across 18,866 records. Compare rates within creation years to reduce the known age confound.

### Expected outcomes and plan

The lab produces a star schema, time and category dimensions, data cubes, OLAP operations, measure and architecture comparisons, a full cube, an iceberg cube, and a research interpretation. It follows CRISP-DM business understanding, data understanding, data preparation, modeling, and evaluation. Deployment belongs to the later research pipeline.

<a name="p2"></a>

---

# Section 02: Data Dictionary
---

## GitHub repository snapshot data dictionary

| Attribute | Role | Meaning |
| --- | --- | --- |
| `repo_id` | Key | Unique GitHub repository ID. |
| `created_at`, `year`, `quarter`, `month` | Time dimension | Creation date and derived hierarchy. |
| `language` | Dimension | Primary programming language. |
| `owner_type` | Dimension | User or organization. |
| `license_group` | Dimension | Grouped license from the previous lab. |
| `maintenance_status` | Dimension | Snapshot label: abandoned, maintained, or unlabeled. |
| `stars`, `forks`, `open_issues`, `size_kb`, `topic_count` | Measures | Repository counts at the snapshot. |

The time hierarchy is **month → quarter → year**. A creation cohort is not a history of its stars or maintenance. The prior lab found that older repositories are more likely to satisfy the abandonment rule, so status comparisons should retain creation year.

<a name="p3"></a>

---

# Section 03: Data Understanding
---

## Step 1: Data Collection

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

output_dir = Path('output')
output_dir.mkdir(exist_ok=True)
print('Imports loaded')

Imports loaded


In [2]:
# Use the cleaned cohort and recover its original snapshot measures and creation timestamp.
data_dir = Path('../DM-Lab')
raw = pd.read_csv(data_dir / 'github_repos_2019_2025.csv', parse_dates=['created_at'])
clean = pd.read_csv(data_dir / 'github_repos_clean.csv',
                    usecols=['repo_id', 'license_group', 'maintenance_status'],
                    keep_default_na=False)
print('Raw shape:', raw.shape, 'Clean shape:', clean.shape)
print('Snapshot dates:', raw['snapshot_date'].unique())
display(raw.head(3))

Raw shape: (25173, 21) Clean shape: (18866, 3)
Snapshot dates: <StringArray>
['2026-09-28']
Length: 1, dtype: str


,repo_id,full_name,owner_type,language,created_at,pushed_at,updated_at,stars,forks,open_issues,...,license,topics,topic_count,description,has_homepage,has_wiki,has_pages,has_discussions,archived,snapshot_date
0,165990457,Jinnrry/getAwayBSG,User,Go,2019-01-16 06:56:14+00:00,2019-09-05T15:10:49Z,2026-09-28T12:56:39Z,1148,116,8,...,AGPL-3.0,NaN,0,逃离北上广,True,True,True,False,True,2026-09-28
1,165957180,pingcap/dm,Organization,Go,2019-01-16 02:13:45+00:00,2022-05-09T08:28:36Z,2026-09-24T09:51:12Z,454,189,127,...,Apache-2.0,hacktoberfest,1,Data Migration Platform,False,True,False,False,True,2026-09-28
2,166091373,assafmo/xioc,User,Go,2019-01-16 18:38:35+00:00,2020-04-19T17:42:40Z,2026-08-10T15:37:37Z,163,9,4,...,MIT,command-line;command-line-tool;data-mining;def...,14,"Extract indicators of compromise from text, in...",False,True,False,False,False,2026-09-28


In [3]:
# The previous lab already saved both source files; verify they exist and identify the snapshot.
assert (data_dir / 'github_repos_2019_2025.csv').is_file()
assert (data_dir / 'github_repos_clean.csv').is_file()
assert raw['snapshot_date'].nunique() == 1
print('Using the saved raw and cleaned datasets')

Using the saved raw and cleaned datasets


## Step 2: Data Description

In [4]:
print('Raw rows:', len(raw), 'Cleaned rows:', len(clean))
print('Raw columns:', raw.columns.tolist())
display(raw.head(3))
display(raw.tail(3))

Raw rows: 25173 Cleaned rows: 18866
Raw columns: ['repo_id', 'full_name', 'owner_type', 'language', 'created_at', 'pushed_at', 'updated_at', 'stars', 'forks', 'open_issues', 'size_kb', 'license', 'topics', 'topic_count', 'description', 'has_homepage', 'has_wiki', 'has_pages', 'has_discussions', 'archived', 'snapshot_date']


,repo_id,full_name,owner_type,language,created_at,pushed_at,updated_at,stars,forks,open_issues,...,license,topics,topic_count,description,has_homepage,has_wiki,has_pages,has_discussions,archived,snapshot_date
0,165990457,Jinnrry/getAwayBSG,User,Go,2019-01-16 06:56:14+00:00,2019-09-05T15:10:49Z,2026-09-28T12:56:39Z,1148,116,8,...,AGPL-3.0,NaN,0,逃离北上广,True,True,True,False,True,2026-09-28
1,165957180,pingcap/dm,Organization,Go,2019-01-16 02:13:45+00:00,2022-05-09T08:28:36Z,2026-09-24T09:51:12Z,454,189,127,...,Apache-2.0,hacktoberfest,1,Data Migration Platform,False,True,False,False,True,2026-09-28
2,166091373,assafmo/xioc,User,Go,2019-01-16 18:38:35+00:00,2020-04-19T17:42:40Z,2026-08-10T15:37:37Z,163,9,4,...,MIT,command-line;command-line-tool;data-mining;def...,14,"Extract indicators of compromise from text, in...",False,True,False,False,False,2026-09-28


,repo_id,full_name,owner_type,language,created_at,pushed_at,updated_at,stars,forks,open_issues,...,license,topics,topic_count,description,has_homepage,has_wiki,has_pages,has_discussions,archived,snapshot_date
25170,1064321320,pix3dev/pix3,Organization,TypeScript,2025-09-25 21:30:17+00:00,2026-09-28T14:32:32Z,2026-09-28T14:32:38Z,10,0,0,...,Apache-2.0,NaN,0,PlayableAds and web games AI Agents first editor,True,True,True,False,False,2026-09-28
25171,1064007526,Haider-Mukhtar/ReactNative-Apple-Health-IOS,User,TypeScript,2025-09-25 12:15:05+00:00,2025-09-27T06:37:38Z,2026-08-06T18:16:13Z,10,2,0,...,MIT,applehealth;expo;ios;react-native;react-native...,6,Integration of Apple Health with React Native ...,False,True,False,False,False,2026-09-28
25172,1063656582,better-auth-extended/dashboard,Organization,TypeScript,2025-09-25 00:05:24+00:00,2025-10-09T22:11:35Z,2026-08-17T21:29:18Z,10,0,0,...,MIT,better-auth;better-auth-admin;better-auth-dash...,4,A Better-Auth powered admin dashboard.,True,False,False,False,False,2026-09-28


## Step 3: Data Exploration

In [5]:
# Frequency tables for the snapshot measures.
for col in ['stars', 'forks', 'open_issues', 'size_kb', 'topic_count']:
    print('Frequency:', col)
    display(pd.cut(raw[col], bins=10).value_counts().sort_index().to_frame('repositories'))

Frequency: stars


,repositories
stars,
"(-100.845, 11094.5]",25134
"(11094.5, 22179.0]",20
"(22179.0, 33263.5]",7
"(33263.5, 44348.0]",5
"(44348.0, 55432.5]",1
"(55432.5, 66517.0]",3
"(66517.0, 77601.5]",1
"(77601.5, 88686.0]",0
"(88686.0, 99770.5]",1


Frequency: forks


,repositories
forks,
"(-25.05, 2505.0]",25145
"(2505.0, 5010.0]",13
"(5010.0, 7515.0]",7
"(7515.0, 10020.0]",5
"(10020.0, 12525.0]",1
"(12525.0, 15030.0]",0
"(15030.0, 17535.0]",1
"(17535.0, 20040.0]",0
"(20040.0, 22545.0]",0


Frequency: open_issues


,repositories
open_issues,
"(-9.433, 943.3]",25161
"(943.3, 1886.6]",7
"(1886.6, 2829.9]",1
"(2829.9, 3773.2]",0
"(3773.2, 4716.5]",0
"(4716.5, 5659.8]",1
"(5659.8, 6603.1]",2
"(6603.1, 7546.4]",0
"(7546.4, 8489.7]",0


Frequency: size_kb


,repositories
size_kb,
"(-27295.925, 2729592.5]",25141
"(2729592.5, 5459185.0]",16
"(5459185.0, 8188777.5]",8
"(8188777.5, 10918370.0]",4
"(10918370.0, 13647962.5]",0
"(13647962.5, 16377555.0]",0
"(16377555.0, 19107147.5]",0
"(19107147.5, 21836740.0]",2
"(21836740.0, 24566332.5]",0


Frequency: topic_count


,repositories
topic_count,
"(-0.02, 2.0]",16102
"(2.0, 4.0]",2966
"(4.0, 6.0]",2466
"(6.0, 8.0]",1446
"(8.0, 10.0]",806
"(10.0, 12.0]",512
"(12.0, 14.0]",288
"(14.0, 16.0]",210
"(16.0, 18.0]",149


## Step 4: Data Verification

In [6]:
print('Raw shape:', raw.shape)
print('Clean shape:', clean.shape)
print('Missing clean keys:', clean['repo_id'].isna().sum())
print('Duplicate clean keys:', clean['repo_id'].duplicated().sum())

Raw shape: (25173, 21)
Clean shape: (18866, 3)
Missing clean keys: 0
Duplicate clean keys: 0


<a name="p4"></a>

---
# Section 04: Data Preparation
___


CRISP-DM preparation: select data, check quality, construct time and comparison attributes, integrate dimensions and facts, format values, and document metadata.

## Step 1: Select Data

In [7]:
# Join one row per cleaned repository to its original snapshot values.
fields = ['repo_id', 'full_name', 'created_at', 'language', 'owner_type',
          'stars', 'forks', 'open_issues', 'size_kb', 'topic_count']
warehouse_data = clean.merge(raw[fields], on='repo_id', validate='one_to_one')
warehouse_data['year'] = warehouse_data['created_at'].dt.year
warehouse_data['quarter'] = 'Q' + warehouse_data['created_at'].dt.quarter.astype(str)
warehouse_data['month'] = warehouse_data['created_at'].dt.month
print('Selected warehouse shape:', warehouse_data.shape)
display(warehouse_data.head())

Selected warehouse shape: (18866, 15)


,repo_id,license_group,maintenance_status,full_name,created_at,language,owner_type,stars,forks,open_issues,size_kb,topic_count,year,quarter,month
0,165957180,Apache-2.0,abandoned,pingcap/dm,2019-01-16 02:13:45+00:00,Go,Organization,454,189,127,23101,1,2019,Q1,1
1,166091373,MIT,abandoned,assafmo/xioc,2019-01-16 18:38:35+00:00,Go,User,163,9,4,66,14,2019,Q1,1
2,165960557,None,abandoned,naqvijafar91/cuteDB,2019-01-16 02:37:59+00:00,Go,User,150,26,5,727,8,2019,Q1,1
3,166055089,MIT,unlabeled,giongto35/gowog,2019-01-16 14:36:47+00:00,Go,User,126,30,0,7951,7,2019,Q1,1
4,166048649,MIT,abandoned,emvi/hide,2019-01-16 13:54:17+00:00,Go,Organization,73,6,1,294,4,2019,Q1,1


### Select data: answers

1. **Which attributes are dimensions?** Creation year, quarter, month, language, owner type, license group, and maintenance status.
2. **Which are measures?** Repository count, stars, forks, open issues, size in KB, and topic count.
3. **Why retain `repo_id` (the original lab's `OrderID`)?** It identifies the fact grain, enables the one-to-one join, and prevents double counting.

## Step 2: Clean Data

In [8]:
print('Missing values in selected warehouse fields:')
print(warehouse_data.isna().sum())

Missing values in selected warehouse fields:
repo_id               0
license_group         0
maintenance_status    0
full_name             0
created_at            0
language              0
owner_type            0
stars                 0
forks                 0
open_issues           0
size_kb               0
topic_count           0
year                  0
quarter               0
month                 0
dtype: int64


In [9]:
print('Duplicate repository IDs:', warehouse_data['repo_id'].duplicated().sum())
print('Fully duplicate rows:', warehouse_data.duplicated().sum())

Duplicate repository IDs: 0
Fully duplicate rows: 0


In [10]:
measures = ['stars', 'forks', 'open_issues', 'size_kb', 'topic_count']
print('Negative measure values:', (warehouse_data[measures] < 0).sum().to_dict())
print('Zero-star records:', (warehouse_data['stars'] == 0).sum())
assert warehouse_data[measures].notna().all().all()
assert (warehouse_data[measures] >= 0).all().all()

Negative measure values: {'stars': 0, 'forks': 0, 'open_issues': 0, 'size_kb': 0, 'topic_count': 0}
Zero-star records: 0


In [11]:
display(warehouse_data[measures].describe().round(1))

,stars,forks,open_issues,size_kb,topic_count
count,18866.0,18866.0,18866.0,18866.0,18866.0
mean,177.4,29.8,8.8,34752.8,3.2
std,1750.5,299.6,79.2,383731.2,4.3
min,10.0,0.0,0.0,1.0,0.0
25%,14.0,2.0,0.0,171.0,0.0
50%,25.0,5.0,1.0,1277.0,0.0
75%,62.0,13.0,6.0,9024.8,5.0
max,110855.0,25050.0,9433.0,27295925.0,20.0


### Clean data: answers

1. **Were missing values found?** No missing values remain in the selected warehouse fields. The previous lab filtered or grouped missing source values. The literal license category `None` is preserved when reading the cleaned CSV.
2. **Were duplicates found?** No duplicate `repo_id` values or duplicate rows remain in the selected warehouse.
3. **Why does quality matter?** Duplicate repositories inflate counts; missing dimensions hide records from groups; invalid measures distort totals.

## Step 3: Construct Data

Create time and comparison attributes from existing repository fields. The synthetic sales measures in the original lab have no repository equivalent.

In [12]:
warehouse_data['time_key'] = (warehouse_data['year'].astype(str) + '_' +
                              warehouse_data['quarter'] + '_' +
                              warehouse_data['month'].astype(str).str.zfill(2))
display(warehouse_data[['created_at', 'time_key']].head())

,created_at,time_key
0,2019-01-16 02:13:45+00:00,2019_Q1_01
1,2019-01-16 18:38:35+00:00,2019_Q1_01
2,2019-01-16 02:37:59+00:00,2019_Q1_01
3,2019-01-16 14:36:47+00:00,2019_Q1_01
4,2019-01-16 13:54:17+00:00,2019_Q1_01


In [13]:
# Equivalent ratio: forks relative to stars; all cleaned repositories have at least 10 stars.
warehouse_data['fork_star_ratio'] = warehouse_data['forks'] / warehouse_data['stars']
display(warehouse_data['fork_star_ratio'].describe().round(3))

count    18866.000
mean         0.280
std          0.994
min          0.000
25%          0.074
50%          0.158
75%          0.300
max         74.347
Name: fork_star_ratio, dtype: float64

In [14]:
warehouse_data['language_owner'] = (warehouse_data['language'] + '_' +
                                     warehouse_data['owner_type'])
display(warehouse_data['language_owner'].value_counts().head())

language_owner
Python_User                6517
JavaScript_User            2491
Python_Organization        2219
TypeScript_User            2153
TypeScript_Organization    1115
Name: count, dtype: int64

In [15]:
display(warehouse_data[['time_key', 'fork_star_ratio', 'language_owner']].head())

,time_key,fork_star_ratio,language_owner
0,2019_Q1_01,0.416300,Go_Organization
1,2019_Q1_01,0.055215,Go_User
2,2019_Q1_01,0.173333,Go_User
3,2019_Q1_01,0.238095,Go_User
4,2019_Q1_01,0.082192,Go_Organization


### Construct data: answers

1. **Why might profit margin be useful in the original sales lab?** It compares profit relative to sales across differently sized groups. Profit margin has no meaning here. The fork-to-star ratio is a repository-specific comparison measure, but it is not itself an unmet-need score.
2. **Why are derived attributes helpful?** Year, quarter, and month expose the creation-time hierarchy; the ratio compares engagement across repositories of different popularity.
3. **What insight can a combined region-product attribute provide here?** `language_owner` identifies an ecosystem and ownership group together, so we can compare their repository counts and maintenance patterns.

## Step 4: Integrate Data

Create one fact table and five dimensions. Each dimension key identifies one member. The fact table has one row per repository at the snapshot.

In [16]:
dim_time = warehouse_data[['time_key', 'year', 'quarter', 'month']].drop_duplicates().sort_values(['year', 'month']).reset_index(drop=True)
dim_time.insert(0, 'time_id', range(1, len(dim_time) + 1))
display(dim_time.head())

,time_id,time_key,year,quarter,month
0,1,2019_Q1_01,2019,Q1,1
1,2,2019_Q1_02,2019,Q1,2
2,3,2019_Q1_03,2019,Q1,3
3,4,2019_Q2_04,2019,Q2,4
4,5,2019_Q2_05,2019,Q2,5


In [17]:
dim_language = warehouse_data[['language']].drop_duplicates().sort_values('language').reset_index(drop=True)
dim_language.insert(0, 'language_id', range(1, len(dim_language) + 1))
display(dim_language)

,language_id,language
0,1,Go
1,2,Java
2,3,JavaScript
3,4,Python
4,5,Rust
5,6,TypeScript


In [18]:
dim_owner = warehouse_data[['owner_type']].drop_duplicates().sort_values('owner_type').reset_index(drop=True)
dim_owner.insert(0, 'owner_id', range(1, len(dim_owner) + 1))
display(dim_owner)

,owner_id,owner_type
0,1,Organization
1,2,User


In [19]:
dim_license = warehouse_data[['license_group']].drop_duplicates().sort_values('license_group').reset_index(drop=True)
dim_license.insert(0, 'license_id', range(1, len(dim_license) + 1))
dim_status = warehouse_data[['maintenance_status']].drop_duplicates().sort_values('maintenance_status').reset_index(drop=True)
dim_status.insert(0, 'status_id', range(1, len(dim_status) + 1))
display(dim_license)
display(dim_status)

,license_id,license_group
0,1,Apache-2.0
1,2,GPL-3.0
2,3,MIT
3,4,None
4,5,Other


,status_id,maintenance_status
0,1,abandoned
1,2,maintained
2,3,unlabeled


In [20]:
fact_repository = warehouse_data[['repo_id', 'full_name', *measures]].copy()
for source_col, dim, key in [
    ('time_key', dim_time, 'time_id'),
    ('language', dim_language, 'language_id'),
    ('owner_type', dim_owner, 'owner_id'),
    ('license_group', dim_license, 'license_id'),
    ('maintenance_status', dim_status, 'status_id'),
]:
    fact_repository = fact_repository.merge(
        warehouse_data[['repo_id', source_col]].merge(dim[[source_col, key]], on=source_col, validate='many_to_one')[['repo_id', key]],
        on='repo_id', validate='one_to_one')
fact_repository['repo_count'] = 1
assert fact_repository['repo_id'].is_unique and len(fact_repository) == len(warehouse_data)
display(fact_repository.head())

,repo_id,full_name,stars,forks,open_issues,size_kb,topic_count,time_id,language_id,owner_id,license_id,status_id,repo_count
0,165957180,pingcap/dm,454,189,127,23101,1,1,1,1,1,1,1
1,166091373,assafmo/xioc,163,9,4,66,14,1,1,2,3,1,1
2,165960557,naqvijafar91/cuteDB,150,26,5,727,8,1,1,2,4,1,1
3,166055089,giongto35/gowog,126,30,0,7951,7,1,1,2,3,3,1
4,166048649,emvi/hide,73,6,1,294,4,1,1,1,3,1,1


### Integrate data: answers

1. **Purpose of a fact table?** Preserve one repository per row, dimension keys, and the snapshot measures.
2. **Why separate dimensions?** They define reusable, consistent group members and clarify each relationship to the fact.
3. **Which table contains measures?** `fact_repository`.

## Step 5: Format Data

Format category and time fields to keep grouping and sorting consistent.

In [21]:
for col in ['language', 'owner_type', 'license_group', 'maintenance_status']:
    warehouse_data[col] = warehouse_data[col].astype('category')
print('Converted categorical columns')

Converted categorical columns


In [22]:
print(warehouse_data.dtypes.to_string())

repo_id                             int64
license_group                    category
maintenance_status               category
full_name                             str
created_at            datetime64[us, UTC]
language                         category
owner_type                       category
stars                               int64
forks                               int64
open_issues                         int64
size_kb                             int64
topic_count                         int64
year                                int32
quarter                               str
month                               int32
time_key                              str
fork_star_ratio                   float64
language_owner                        str


In [23]:
warehouse_data = warehouse_data.sort_values(['year', 'month', 'repo_id']).reset_index(drop=True)
display(warehouse_data[['repo_id', 'year', 'quarter', 'month']].head())

,repo_id,year,quarter,month
0,165942605,2019,Q1,1
1,165945113,2019,Q1,1
2,165946267,2019,Q1,1
3,165946956,2019,Q1,1
4,165947272,2019,Q1,1


In [24]:
prepared_path = output_dir / 'warehouse_repositories_prepared.csv'
warehouse_data.to_csv(prepared_path, index=False)
print('Saved', prepared_path, 'rows:', len(warehouse_data))

Saved output/warehouse_repositories_prepared.csv rows: 18866


### Format data: answers

1. **Why use categorical types?** Repeated labels can use less memory and make the intended groups explicit.
2. **Why format consistently?** Different spellings or cases of a language or status would split one group into several.
3. **How does formatting improve OLAP?** Consistent keys make joins correct, and numeric year/month fields sort in calendar order.

## Step 6: Metadata Repository

Metadata records the meaning, role, table, and aggregation rule for each warehouse attribute. It prevents a snapshot total from being mistaken for a historical trend.

In [25]:
metadata_repository = pd.DataFrame([
    ('repo_id', 'FactRepository', 'Key', 'GitHub repository ID'),
    ('time_id', 'FactRepository', 'Foreign key', 'Creation-time dimension member'),
    ('language_id', 'FactRepository', 'Foreign key', 'Primary language member'),
    ('owner_id', 'FactRepository', 'Foreign key', 'Owner type member'),
    ('license_id', 'FactRepository', 'Foreign key', 'License group member'),
    ('status_id', 'FactRepository', 'Foreign key', 'Snapshot maintenance label member'),
    ('year', 'DimTime', 'Dimension', 'Creation year'),
    ('quarter', 'DimTime', 'Dimension', 'Creation quarter'),
    ('month', 'DimTime', 'Dimension', 'Creation month number'),
    ('language', 'DimLanguage', 'Dimension', 'Primary programming language'),
    ('owner_type', 'DimOwner', 'Dimension', 'User or organization'),
    ('license_group', 'DimLicense', 'Dimension', 'Grouped license'),
    ('maintenance_status', 'DimStatus', 'Dimension', 'Status at snapshot'),
    ('repo_count', 'FactRepository', 'Distributive measure', 'One per repository; sum across disjoint groups'),
    ('stars', 'FactRepository', 'Distributive measure', 'Stars at snapshot; sum across disjoint groups'),
    ('forks', 'FactRepository', 'Distributive measure', 'Forks at snapshot; sum across disjoint groups'),
    ('open_issues', 'FactRepository', 'Distributive measure', 'Open issues at snapshot; sum across disjoint groups'),
    ('size_kb', 'FactRepository', 'Distributive measure', 'Size at snapshot in KB'),
    ('topic_count', 'FactRepository', 'Distributive measure', 'Number of topics at snapshot'),
], columns=['Attribute', 'TableName', 'Type', 'Description'])
display(metadata_repository)

,Attribute,TableName,Type,Description
0,repo_id,FactRepository,Key,GitHub repository ID
1,time_id,FactRepository,Foreign key,Creation-time dimension member
2,language_id,FactRepository,Foreign key,Primary language member
3,owner_id,FactRepository,Foreign key,Owner type member
4,license_id,FactRepository,Foreign key,License group member
5,status_id,FactRepository,Foreign key,Snapshot maintenance label member
6,year,DimTime,Dimension,Creation year
7,quarter,DimTime,Dimension,Creation quarter
8,month,DimTime,Dimension,Creation month number
9,language,DimLanguage,Dimension,Primary programming language


In [26]:
metadata_path = output_dir / 'metadata_repository.csv'
metadata_repository.to_csv(metadata_path, index=False)
print('Saved', metadata_path)

Saved output/metadata_repository.csv


In [27]:
display(metadata_repository[metadata_repository['Type'].str.contains('measure')])

,Attribute,TableName,Type,Description
13,repo_count,FactRepository,Distributive measure,One per repository; sum across disjoint groups
14,stars,FactRepository,Distributive measure,Stars at snapshot; sum across disjoint groups
15,forks,FactRepository,Distributive measure,Forks at snapshot; sum across disjoint groups
16,open_issues,FactRepository,Distributive measure,Open issues at snapshot; sum across disjoint g...
17,size_kb,FactRepository,Distributive measure,Size at snapshot in KB
18,topic_count,FactRepository,Distributive measure,Number of topics at snapshot


In [28]:
display(metadata_repository[metadata_repository['Type'] == 'Dimension'])

,Attribute,TableName,Type,Description
6,year,DimTime,Dimension,Creation year
7,quarter,DimTime,Dimension,Creation quarter
8,month,DimTime,Dimension,Creation month number
9,language,DimLanguage,Dimension,Primary programming language
10,owner_type,DimOwner,Dimension,User or organization
11,license_group,DimLicense,Dimension,Grouped license
12,maintenance_status,DimStatus,Dimension,Status at snapshot


In [29]:
display(metadata_repository[metadata_repository['Attribute'] == 'stars'])

,Attribute,TableName,Type,Description
14,stars,FactRepository,Distributive measure,Stars at snapshot; sum across disjoint groups


In [30]:
display(metadata_repository[metadata_repository['Attribute'] == 'maintenance_status'])

,Attribute,TableName,Type,Description
12,maintenance_status,DimStatus,Dimension,Status at snapshot


In [31]:
warehouse_metadata = pd.DataFrame([
    ('FactRepository', 'Fact', len(fact_repository)),
    ('DimTime', 'Dimension', len(dim_time)),
    ('DimLanguage', 'Dimension', len(dim_language)),
    ('DimOwner', 'Dimension', len(dim_owner)),
    ('DimLicense', 'Dimension', len(dim_license)),
    ('DimStatus', 'Dimension', len(dim_status)),
], columns=['TableName', 'TableType', 'Rows'])
display(warehouse_metadata)

,TableName,TableType,Rows
0,FactRepository,Fact,18866
1,DimTime,Dimension,81
2,DimLanguage,Dimension,6
3,DimOwner,Dimension,2
4,DimLicense,Dimension,5
5,DimStatus,Dimension,3


### Metadata: answers

1. **Purpose?** Document each field's meaning, location, and aggregation rule.
2. **How does it improve management?** Analysts can interpret reports consistently and audit which snapshot and categories produced them.
3. **Which attributes are dimensions?** Creation year, quarter, month, language, owner type, license group, and maintenance status.
4. **Which are measures?** Repository count, stars, forks, open issues, size, and topic count.
5. **Why important to business intelligence?** It prevents reports from silently changing grain or treating snapshot counts as longitudinal trends.

## Data preparation outcome

The cleaned cohort contains one row per repository, passes the quality checks, has a creation-time hierarchy and derived comparison fields, and has a fact table, five dimensions, and saved warehouse and metadata CSVs.

<a name="p5"></a>

---

# Section 05: Modeling
---

The modeling phase constructs repository cubes and OLAP views rather than a predictive model. The research proposal's text clustering, abandonment classifier, and ranking are later project work.

## Step 1: Dimensions and Measures

In [32]:
dimensions = ['year', 'quarter', 'month', 'language', 'owner_type', 'license_group', 'maintenance_status']
print('Dimensions:', dimensions)

Dimensions: ['year', 'quarter', 'month', 'language', 'owner_type', 'license_group', 'maintenance_status']


In [33]:
measures = ['repo_count', 'stars', 'forks', 'open_issues', 'size_kb', 'topic_count']
print('Measures:', measures)

Measures: ['repo_count', 'stars', 'forks', 'open_issues', 'size_kb', 'topic_count']


### Dimensions and measures: answers

1. **What describes the context of a sale, translated here?** A repository's creation time, language, owner type, license, and maintenance status.
2. **What can be aggregated?** Repository counts and the snapshot count measures across disjoint repository groups. Median stars requires underlying rows.
3. **Why are measures in the fact table?** They belong to individual repositories and vary independently of the dimension labels.

## Step 2: Star schema

```text
DimTime ───────┐
DimLanguage ────┤
DimOwner ───────┼── FactRepository (one repository at one snapshot)
DimLicense ─────┤
DimStatus ──────┘
```

A **snowflake** could normalize the time hierarchy as `DimMonth → DimQuarter → DimYear`. A **fact constellation** would share `DimRepository` and `DimTime` between a repository-snapshot fact and an issue-event fact. The latter is only a proposed extension; this dataset has no issue-event rows or repeated snapshots.

In [34]:
print('Fact rows:', len(fact_repository))
display(fact_repository.head())

Fact rows: 18866


,repo_id,full_name,stars,forks,open_issues,size_kb,topic_count,time_id,language_id,owner_id,license_id,status_id,repo_count
0,165957180,pingcap/dm,454,189,127,23101,1,1,1,1,1,1,1
1,166091373,assafmo/xioc,163,9,4,66,14,1,1,2,3,1,1
2,165960557,naqvijafar91/cuteDB,150,26,5,727,8,1,1,2,4,1,1
3,166055089,giongto35/gowog,126,30,0,7951,7,1,1,2,3,3,1
4,166048649,emvi/hide,73,6,1,294,4,1,1,1,3,1,1


In [35]:
for name, dim in [('time', dim_time), ('language', dim_language),
                  ('owner', dim_owner), ('license', dim_license), ('status', dim_status)]:
    print(name, 'members:', len(dim))
    display(dim.head())

time members: 81


,time_id,time_key,year,quarter,month
0,1,2019_Q1_01,2019,Q1,1
1,2,2019_Q1_02,2019,Q1,2
2,3,2019_Q1_03,2019,Q1,3
3,4,2019_Q2_04,2019,Q2,4
4,5,2019_Q2_05,2019,Q2,5


language members: 6


,language_id,language
0,1,Go
1,2,Java
2,3,JavaScript
3,4,Python
4,5,Rust


owner members: 2


,owner_id,owner_type
0,1,Organization
1,2,User


license members: 5


,license_id,license_group
0,1,Apache-2.0
1,2,GPL-3.0
2,3,MIT
3,4,None
4,5,Other


status members: 3


,status_id,maintenance_status
0,1,abandoned
1,2,maintained
2,3,unlabeled


### Star schema: answers

1. **Purpose of fact table?** Preserve the repository grain, foreign keys, and measures.
2. **Dimension tables contain?** Unique time, language, owner, license, and status members with surrogate keys.
3. **Why useful?** The same dimensions support many consistent cohort queries.

## Step 3: Data Cube

In [36]:
cube_language_owner = pd.pivot_table(warehouse_data, values='repo_id',
                                     index='language', columns='owner_type',
                                     aggfunc='count', fill_value=0, margins=True,
                                     observed=True)
assert cube_language_owner.iloc[-1, -1] == len(warehouse_data)
display(cube_language_owner)

owner_type,Organization,User,All
language,,,
Go,554,825,1379
Java,319,777,1096
JavaScript,731,2491,3222
Python,2219,6517,8736
Rust,376,789,1165
TypeScript,1115,2153,3268
All,5314,13552,18866


In [37]:
cube_3d = pd.pivot_table(warehouse_data, values='repo_id',
                         index=['language', 'owner_type'], columns='year',
                         aggfunc='count', fill_value=0, margins=True,
                         observed=True)
assert cube_3d.iloc[-1, -1] == len(warehouse_data)
display(cube_3d)

year                     2019  2020  2021  2022  2023  2024  2025    All
language   owner_type                                                   
Go         Organization    99   115    92    64    66    67    51    554
           User           146   169   129   109   101    97    74    825
Java       Organization    69    75    60    43    40    18    14    319
           User           170   163   142   104    94    65    39    777
JavaScript Organization   170   193   134    89    67    40    38    731
           User           517   480   417   346   292   235   204   2491
Python     Organization   240   341   291   292   341   351   363   2219
           User           875  1071   912   939  1001   942   777   6517
Rust       Organization    36    46    51    46    82    56    59    376
           User            81   121   115   118   126   116   112    789
TypeScript Organization   124   157   169   182   192   134   157   1115
           User           236   317   308   319   341   335   297   2153
All                      2763  3248  2820  2651  2743  2456  2185  18866

### Data cube: answers

1. **Which dimensions define the cube?** Language, owner type, and creation year.
2. **Which measure is summarized?** Repository count.
3. **What insight?** It shows cohort size by ecosystem, ownership, and year. Python has 8,736 repositories overall: 6,517 user-owned and 2,219 organization-owned. That is a count difference, not evidence of unmet demand.

## Step 4: OLAP Operations

In [38]:
rollup = warehouse_data.groupby(['year', 'quarter'], observed=True).size().rename('repositories')
display(rollup.to_frame())

repositories
year quarter              
2019 Q1                723
     Q2                756
     Q3                665
     Q4                619
2020 Q1                745
     Q2                941
     Q3                847
     Q4                715
2021 Q1                810
     Q2                811
     Q3                603
     Q4                596
2022 Q1                610
     Q2                752
     Q3                671
     Q4                618
2023 Q1                699
     Q2                791
     Q3                630
     Q4                623
2024 Q1                650
     Q2                646
     Q3                517
     Q4                643
2025 Q1                661
     Q2                836
     Q3                688

In [39]:
drilldown = warehouse_data.groupby(['year', 'quarter', 'month'], observed=True).size().rename('repositories')
assert rollup.equals(drilldown.groupby(level=['year', 'quarter']).sum())
display(drilldown.to_frame().head(16))

repositories
year quarter month              
2019 Q1      1               245
             2               220
             3               258
     Q2      4               245
             5               193
             6               318
     Q3      7               234
             8               201
             9               230
     Q4      10              180
             11              262
             12              177
2020 Q1      1               220
             2               269
             3               256
     Q2      4               323

In [40]:
slice_python = warehouse_data.loc[warehouse_data['language'].eq('Python')]
print('Python repositories:', len(slice_python))
display(slice_python.head())

Python repositories: 8736


,repo_id,license_group,maintenance_status,full_name,created_at,language,owner_type,stars,forks,open_issues,size_kb,topic_count,year,quarter,month,time_key,fork_star_ratio,language_owner
0,165942605,MIT,unlabeled,ThiagoGrabe/GeneticAlgorithm-PathPlanning,2019-01-16 00:00:20+00:00,Python,User,12,1,0,8468,0,2019,Q1,1,2019_Q1_01,0.083333,Python_User
4,165947272,None,abandoned,nyavramov/fast_scene_detection,2019-01-16 00:50:03+00:00,Python,User,52,2,3,26,7,2019,Q1,1,2019_Q1_01,0.038462,Python_User
5,165948182,MIT,unlabeled,vsitzmann/pytorch_prototyping,2019-01-16 00:58:58+00:00,Python,User,65,14,0,26,0,2019,Q1,1,2019_Q1_01,0.215385,Python_User
6,165948405,None,abandoned,hgiesel/anki-plus-archive,2019-01-16 01:01:12+00:00,Python,User,14,0,16,1481,3,2019,Q1,1,2019_Q1_01,0.000000,Python_User
9,165949101,None,unlabeled,OlivierLaflamme/CTF-Script-And-Template-Thrift...,2019-01-16 01:07:47+00:00,Python,User,63,15,0,356,20,2019,Q1,1,2019_Q1_01,0.238095,Python_User


In [41]:
dice = warehouse_data.loc[
    warehouse_data['language'].isin(['Python', 'TypeScript']) &
    warehouse_data['owner_type'].eq('User') &
    warehouse_data['year'].between(2022, 2024)]
print('Selected repositories:', len(dice))
display(dice[['repo_id', 'language', 'owner_type', 'year']].head())

Selected repositories: 3877


,repo_id,language,owner_type,year
8833,445683525,TypeScript,User,2022
8836,445688729,Python,User,2022
8837,445689164,TypeScript,User,2022
8841,445692537,Python,User,2022
8842,445695412,Python,User,2022


In [42]:
pivot = pd.pivot_table(warehouse_data, values='repo_id',
                       index='language', columns='maintenance_status',
                       aggfunc='count', fill_value=0, observed=True)
display(pivot)

maintenance_status,abandoned,maintained,unlabeled
language,,,
Go,512,409,458
Java,429,216,451
JavaScript,1576,429,1217
Python,3836,1158,3742
Rust,403,324,438
TypeScript,1318,791,1159


### OLAP operations: answers

1. **How does roll-up change detail?** It sums month-level creation counts into quarters.
2. **How does drill-down increase detail?** It shows the months within each quarter; the code checks that they sum back to the quarter total.
3. **Slice versus dice?** A slice fixes one dimension (`Python`). Dice filters multiple dimensions (language, owner, and year).
4. **Why pivot?** It places language and maintenance status on two axes for direct comparison.

## Step 5: Classify Measures

In [43]:
measure_types = pd.DataFrame({
    'Measure': ['SUM', 'COUNT', 'AVG', 'MEDIAN'],
    'Category': ['Distributive', 'Distributive', 'Algebraic', 'Holistic']})
display(measure_types)

,Measure,Category
0,SUM,Distributive
1,COUNT,Distributive
2,AVG,Algebraic
3,MEDIAN,Holistic


### Measure classification: answers

1. **Why is SUM distributive?** Totals from disjoint repository groups add to the combined total.
2. **Why is AVG algebraic?** Combine subgroup sums and counts to get the overall mean; subgroup means alone are insufficient.
3. **Why is MEDIAN holistic?** Subgroup medians cannot determine the combined median.

## Step 6: Compare OLAP Architectures

In [44]:
olap_architectures = pd.DataFrame({
    'Architecture': ['ROLAP', 'MOLAP', 'HOLAP'],
    'Storage': ['Relational', 'Multidimensional', 'Hybrid'],
    'Typical strength': ['Detailed queries at scale', 'Fast precomputed cube queries', 'Detail plus common aggregates']})
display(olap_architectures)

,Architecture,Storage,Typical strength
0,ROLAP,Relational,Detailed queries at scale
1,MOLAP,Multidimensional,Fast precomputed cube queries
2,HOLAP,Hybrid,Detail plus common aggregates


### OLAP architecture: answers

1. **Which is most scalable?** ROLAP is usually best suited to large detailed relational facts, though capacity depends on the system and workload.
2. **Which responds fastest?** MOLAP can be fastest for precomputed cube queries. There is no universal winner for every query.
3. **When prefer HOLAP?** When analysts need both detailed repository rows and fast common aggregates.

Bitmap indexes fit low-cardinality status or owner fields, join indexes can speed recurring fact-to-dimension joins, and column storage helps scans of a few measures. This pandas lab does not create database indexes.

## Step 7: Cube Computation Methods

In [45]:
cube_methods = pd.DataFrame({
    'Method': ['Multi-way aggregation', 'BUC'],
    'Approach': ['Share work across grouping sets', 'Bottom-up computation with pruning'],
    'Best use': ['Full cube', 'Selective iceberg cube']})
display(cube_methods)

,Method,Approach,Best use
0,Multi-way aggregation,Share work across grouping sets,Full cube
1,BUC,Bottom-up computation with pruning,Selective iceberg cube


### Cube computation: answers

1. **Which method suits a full cube?** Multi-way aggregation shares work across grouping sets.
2. **Which supports an iceberg cube?** BUC can prune groups that fail minimum support.
3. **Why useful with many dimensions?** Pruning avoids materializing many low-support cells when the threshold is selective.

## Modeling outcome

The lab identifies dimensions and measures, constructs a star schema and cubes, performs five OLAP operations, and compares measure types, architectures, and cube methods.

# Section 06: Evaluation

Assess repository cohorts by creation year and maintenance status. The previous lab found that the status label is age-confounded, so compare within years and do not interpret the table causally.

In [46]:
evaluation = warehouse_data.groupby(['year', 'maintenance_status'], observed=True).agg(
    repositories=('repo_id', 'size'),
    median_stars=('stars', 'median'),
    total_open_issues=('open_issues', 'sum')).reset_index()
evaluation['share_in_year_pct'] = (
    100 * evaluation['repositories'] /
    evaluation.groupby('year')['repositories'].transform('sum')).round(1)
assert evaluation['repositories'].sum() == len(warehouse_data)
display(evaluation)

,year,maintenance_status,repositories,median_stars,total_open_issues,share_in_year_pct
0,2019,abandoned,1702,28.0,19230,61.6
1,2019,maintained,333,64.0,9625,12.1
2,2019,unlabeled,728,20.0,2412,26.3
3,2020,abandoned,1800,27.0,15706,55.4
4,2020,maintained,449,51.0,13909,13.8
5,2020,unlabeled,999,19.0,4227,30.8
6,2021,abandoned,1285,29.0,9157,45.6
7,2021,maintained,455,41.0,21185,16.1
8,2021,unlabeled,1080,19.0,2390,38.3
9,2022,abandoned,1109,28.0,7370,41.8


# Section 07: More Data Cube Computations

Compute all grouping sets over language, owner type, and maintenance status. `ALL` means that dimension was rolled up. The result is a full cube of nonempty cells.

In [47]:
from itertools import combinations
cube_dimensions = ['language', 'owner_type', 'maintenance_status']
parts = []
for size in range(len(cube_dimensions) + 1):
    for group in combinations(cube_dimensions, size):
        if group:
            part = warehouse_data.groupby(list(group), observed=True).size().reset_index(name='repositories')
        else:
            part = pd.DataFrame({'repositories': [len(warehouse_data)]})
        for column in cube_dimensions:
            if column not in group:
                part[column] = 'ALL'
        parts.append(part[[*cube_dimensions, 'repositories']])
full_cube = pd.concat(parts, ignore_index=True)
assert full_cube.loc[(full_cube[cube_dimensions] == 'ALL').all(axis=1), 'repositories'].item() == len(warehouse_data)
print('Nonempty cube cells:', len(full_cube))
display(full_cube.head(12))

Nonempty cube cells: 84


,language,owner_type,maintenance_status,repositories
0,ALL,ALL,ALL,18866
1,Go,ALL,ALL,1379
2,Java,ALL,ALL,1096
3,JavaScript,ALL,ALL,3222
4,Python,ALL,ALL,8736
5,Rust,ALL,ALL,1165
6,TypeScript,ALL,ALL,3268
7,ALL,Organization,ALL,5314
8,ALL,User,ALL,13552
9,ALL,ALL,abandoned,8074


# Section 08: Iceberg Cube

Keep language × owner × maintenance cells containing at least 100 repositories. This code aggregates then filters; BUC would prune during computation.

In [48]:
threshold = 100
iceberg = full_cube.loc[
    full_cube[cube_dimensions].ne('ALL').all(axis=1) &
    full_cube['repositories'].ge(threshold)
].sort_values('repositories', ascending=False).reset_index(drop=True)
print('Cells meeting threshold:', len(iceberg))
display(iceberg)

Cells meeting threshold: 35


,language,owner_type,maintenance_status,repositories
0,Python,User,unlabeled,2971
1,Python,User,abandoned,2932
2,JavaScript,User,abandoned,1175
3,JavaScript,User,unlabeled,1048
4,Python,Organization,abandoned,904
5,TypeScript,User,unlabeled,883
6,TypeScript,User,abandoned,879
7,Python,Organization,unlabeled,771
8,Python,User,maintained,614
9,Python,Organization,maintained,544


# Section 09: Conclusion

The warehouse makes creation-time, language, ownership, license, and maintenance summaries reproducible. The data are one repository snapshot, so they cannot show change in stars or issue outcomes over time. The label's age confound and skewed star counts make year-stratified rates and median stars safer than pooled comparisons.

In [49]:
print('Repositories:', len(warehouse_data))
print('Snapshot:', raw['snapshot_date'].iloc[0])
print('Maintenance labels:', warehouse_data['maintenance_status'].value_counts().to_dict())
print('Median stars:', warehouse_data['stars'].median())

Repositories: 18866
Snapshot: 2026-09-28
Maintenance labels: {'abandoned': 8074, 'unlabeled': 7465, 'maintained': 3327}
Median stars: 25.0


# Section 10: Assignment

1. **Cohort dimensions and measures?** Time, language, owner, license, and status define cohorts. Count, stars, forks, issues, size, and topics describe them.
2. **How does abandoned share vary by year?** Among all cleaned repositories, it falls from 61.6% for 2019 creation to 17.8% for 2025. Older repositories have had more time to meet the inactivity rule; this is not proof that newer projects are intrinsically healthier.
3. **Compare Python owners.** The cube has 6,517 user-owned and 2,219 organization-owned Python repositories overall. Inspect year columns before interpreting differences.
4. **Change the iceberg threshold.** Raising it removes smaller cells; lowering it restores them. The source records do not change.
5. **Why no historical fact constellation?** This input has one repository snapshot and no issue-event fact rows. Those facts need to be collected first.

In [50]:
# Final checks for the adapted lab.
assert len(fact_repository) == 18866
assert len(warehouse_data) == len(fact_repository)
assert fact_repository['repo_id'].is_unique
assert warehouse_data[['repo_id', 'language', 'owner_type', 'maintenance_status']].notna().all().all()
assert rollup.sum() == len(warehouse_data)
assert prepared_path.is_file() and metadata_path.is_file()
print('All warehouse checks passed')

All warehouse checks passed


## Deliverable: use in the research project

This lab gives the research team a repeatable way to summarize its fixed GitHub snapshot before clustering and ranking potential unmet developer needs. The repository fact table and shared dimensions make it possible to compare languages, owner types, creation cohorts, and maintenance labels while tracking cohort sizes, stars, and open issues. Those summaries can guide which parts of the corpus need closer inspection and can later supply transparent counts for the ranked report proposed in *Mining GitHub for Unmet Developer Needs*.

The warehouse is groundwork, not the project's final evidence of unmet demand. The proposal calls for clustering repository descriptions, README text, and issue titles by problem; measuring demand and project health; and having developers assess the top candidates. This lab's one repository snapshot lacks issue-event history and repeated observations, and its abandonment label is influenced by repository age. The planned issue and activity data could extend the model with event facts and stronger health measures, while the current cubes remain useful for checking the corpus and explaining the final rankings.